# 01 — Load MS-MARCO Passage (full corpus + dev queries)

Designed to run **once on Colab** so the ~3 GB corpus lands in your Drive cache. After this, any other notebook (this session or future ones) loads instantly from cache.

Outputs:
- `docs_ds` — handle to the 8.8M-passage corpus (`docs_iter()`, `docs_store()`).
- `dev_ds` — dev/small split (6,980 queries + 7,437 qrels).

Indexing / BM25 / evaluation lives in `01_bm25.ipynb`.

In [ ]:
# Universal bootstrap — same in every notebook.
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    %cd /content
    if not os.path.exists('Information-Retreival'):
        !git clone https://github.com/ZiaadNegm/Information-Retreival.git
    %cd Information-Retreival
    %pip install -q -r requirements.txt
else:
    sys.path.insert(0, os.path.abspath('..'))

from bootstrap import setup_env
setup_env()

In [ ]:
# Load handles (cheap — no download yet, just metadata).
import ir_datasets

docs_ds = ir_datasets.load('msmarco-passage')             # 8,841,823 passages
dev_ds  = ir_datasets.load('msmarco-passage/dev/small')   # 6,980 queries + qrels

print('docs   :', docs_ds.docs_count())
print('queries:', dev_ds.queries_count())
print('qrels  :', dev_ds.qrels_count())

In [ ]:
# Force the corpus to materialize. First call streams ~1 GB tar.gz from Microsoft
# and extracts to the random-access store (~3 GB on disk). Expect 5-15 min on Colab.
# Subsequent runs: instant.
first = next(docs_ds.docs_iter())
print('schema :', type(first).__name__, first._fields)
print('sample :', first.doc_id, '|', first.text[:200])

In [ ]:
# Build the random-access store explicitly. Indexing it once means any later notebook
# can do `store.get(doc_id)` in O(1).
store = docs_ds.docs_store()
print('lookup test —', store.get('0').text[:200])

In [ ]:
# Sample query + its qrels.
sample_q = next(iter(dev_ds.queries_iter()))
print('query   :', sample_q.query_id, '|', sample_q.text)
for qrel in dev_ds.qrels_iter():
    if qrel.query_id == sample_q.query_id:
        print('qrel    :', qrel)
        break

In [ ]:
# Disk usage check — confirms data landed in the expected cache dir.
cache = os.environ.get('IR_DATASETS_HOME', os.path.expanduser('~/.ir_datasets'))
print('cache:', cache)
!du -sh "$cache" 2>/dev/null || true